In [6]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [7]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

zeroslope_taxi_data_path = kagglehub.dataset_download('zeroslope/taxi-data')

print('Data source import complete.')


Data source import complete.


# NYC Taxi Fare – Colab Notebook

This notebook runs the full assignment workflow against the random 10k sample (`data/train_10k_sample.csv`).
It is tailored for Google Colab: run the setup cells, upload the dataset into the `data/` folder, and execute the
sections in order to reproduce the experiments and plots required by the rubric.

## 0. Quick start (Colab)

1. Switch Colab runtime to GPU (or TPU) via **Runtime → Change runtime type** if you want accelerator comparisons.
2. Run the cell below to install the pinned dependencies inside the Colab VM.
3. Upload `train_10k_sample.csv` into a `/content/data/` directory (the next cell helps create + verify it).
4. Optionally mount Google Drive if you prefer to read/write artifacts there.

In [8]:
# !export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax
# !export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system tensorflow-tpu=="2.18.0" --find-links https://storage.googleapis.com/libtpu-tf-releases/index.html
# import tensorflow as tf
# print('TensorFlow version' + tf.__version__)
# import tensorflow as tf



In [9]:
# !pip install -U -f https://storage.googleapis.com/libtpu-tf-releases/index.html libtpu


In [10]:
# !pip install --upgrade pip
# !pip install -q numpy
# !pip install -q jax
# !pip install -q pandas
# !pip install -q scikit-learn
# !pip install -q matplotlib
# !pip install -q tensorboard
# !pip install -q tensorflow
!pip install -q tqdm
# !pip install -q --upgrade tensorflow==2.18.0 tf-keras==2.18.0 tensorboard==2.18.0

In [11]:
# !pip install -q --upgrade "numpy>=2.0" "jax>=0.4.30"

### Optional: mount Google Drive
Uncomment the next two lines if your dataset lives on Drive.

In [12]:
# from google.colab import drive
# /kaggle/input/taxi-data/train_10k_sample.csv
# drive.mount('/content/drive')

## 1. Environment & paths
Creates the `data/` directory (if needed) and checks for the sampled CSV.

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras

np.random.seed(7)
tf.random.set_seed(7)
plt.style.use('seaborn-v0_8-darkgrid')

PROJECT_ROOT =  Path('/kaggle/working/')
data_dir = Path(zeroslope_taxi_data_path)
DATA_PATH = data_dir / 'train_100k_sample.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} not found in Kaggle input mount.")
print(f'Using dataset: {DATA_PATH}')

## 2. Helper functions
Preprocessing filters, feature engineering, model factory, and plotting utilities used throughout the notebook.

In [ ]:
FARE_RANGE = (0, 200)
LON_RANGE = (-80, -70)
LAT_RANGE = (35, 45)
PASSENGER_RANGE = (1, 6)
FEATURE_COLUMNS = [
    'pickup_timestamp',
    'pickup_longitude',
    'pickup_latitude',
    'dropoff_longitude',
    'dropoff_latitude',
    'passenger_count',
]
TARGET_COLUMN = 'fare_amount'
DATETIME_COLUMN = 'pickup_datetime'

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.dropna().copy()
    cleaned = cleaned[cleaned[TARGET_COLUMN].between(*FARE_RANGE)]
    cleaned = cleaned[cleaned['pickup_longitude'].between(*LON_RANGE)]
    cleaned = cleaned[cleaned['dropoff_longitude'].between(*LON_RANGE)]
    cleaned = cleaned[cleaned['pickup_latitude'].between(*LAT_RANGE)]
    cleaned = cleaned[cleaned['dropoff_latitude'].between(*LAT_RANGE)]
    cleaned = cleaned[cleaned['passenger_count'].between(*PASSENGER_RANGE)]
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan).dropna()
    return cleaned

def add_pickup_timestamp(df: pd.DataFrame) -> pd.DataFrame:
    enriched = df.copy()
    enriched['pickup_timestamp'] = (
        pd.to_datetime(enriched[DATETIME_COLUMN]).astype('int64') // 10**9
    ).astype('float32')
    return enriched

def build_feature_matrix(df: pd.DataFrame):
    scaler = StandardScaler()
    features = scaler.fit_transform(df[FEATURE_COLUMNS].astype('float32'))
    labels = df[TARGET_COLUMN].astype('float32').to_numpy()
    return features, labels, scaler

def make_datasets(features, labels, batch_size=256, val_split=0.2):
    X_train, X_val, y_train, y_val = train_test_split(
        features, labels, test_size=val_split, random_state=7
    )
    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(len(X_train))
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((X_val, y_val))
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    return train_ds, val_ds, X_train.shape[1]

def build_model(input_dim, hidden_units=(128, 64, 32), dropout=0.1, lr=1e-3):
    model = keras.Sequential(name="taxi_fare_regressor")
    model.add(keras.layers.InputLayer(shape=(input_dim,)))
    for idx, units in enumerate(hidden_units):
        model.add(keras.layers.Dense(units, activation="relu", name=f"dense_{idx}"))
        if dropout:
            model.add(keras.layers.Dropout(dropout, name=f"dropout_{idx}"))
    model.add(keras.layers.Dense(1, name="fare_output"))
    optimizer = keras.optimizers.Adam(learning_rate=lr)  # no “legacy” namespace
    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=[
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model

# def build_model(input_dim, hidden_units=(128, 64, 32), dropout=0.1, lr=1e-3, momentum=0.9):
#     model = keras.Sequential(name="taxi_fare_regressor")
#     model.add(keras.layers.InputLayer(shape=(input_dim,)))
#     for idx, units in enumerate(hidden_units):
#         model.add(keras.layers.Dense(units, activation="relu", name=f"dense_{idx}"))
#         if dropout:
#             model.add(keras.layers.Dropout(dropout, name=f"dropout_{idx}"))
#     model.add(keras.layers.Dense(1, name="fare_output"))
#     optimizer = keras.optimizers.SGD(learning_rate=lr, momentum=momentum, nesterov=True)
#     model.compile(
#         optimizer=optimizer,
#         loss="mse",
#         metrics=[
#             keras.metrics.MeanAbsoluteError(name="mae"),
#             keras.metrics.RootMeanSquaredError(name="rmse"),
#         ],
#     )
#     return model



def format_label(device: str, hidden_units) -> str:
    return f"{device.lower()}-{'x'.join(map(str, hidden_units))}"

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='train')
    axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title('Mean Squared Error (USD²) over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Mean Squared Error (USD²)')
    axes[0].legend()
    axes[1].plot(history.history['mae'], label='train')
    axes[1].plot(history.history['val_mae'], label='val')
    axes[1].set_title('Mean Absolute Error (USD) over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Mean Absolute Error (USD)')
    axes[1].legend()
    fig.suptitle(title)
    plt.show()

def plot_bar(metric_values, ylabel, title):
    ax = sns.barplot(x=list(metric_values.keys()), y=list(metric_values.values()), palette='viridis')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f')
    plt.xticks(rotation=30)
    plt.show()

def plot_tradeoff(points):
    fig, ax = plt.subplots(figsize=(6, 4))
    for label, (epoch_time, mae) in points.items():
        ax.scatter(epoch_time, mae)
        ax.annotate(label, (epoch_time, mae), textcoords='offset points', xytext=(5, -5))
    ax.set_xlabel('Average Epoch Duration (seconds)')
    ax.set_ylabel('Mean Absolute Error (USD)')
    ax.set_title('Epoch Duration vs Validation Mean Absolute Error')
    plt.show()

## 3. Load & preprocess the 10k sample

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
print(f'Raw rows: {len(raw_df):,}')
raw_df[DATETIME_COLUMN] = pd.to_datetime(raw_df[DATETIME_COLUMN])
cleaned = clean_dataframe(raw_df)
enriched = add_pickup_timestamp(cleaned)
features, labels, scaler = build_feature_matrix(enriched)
train_ds, val_ds, input_dim = make_datasets(features, labels)
print(f'Clean rows: {len(enriched):,}')
print('Feature matrix shape:', features.shape)
print('Label vector shape:', labels.shape)
enriched.describe()[[TARGET_COLUMN]].T

## 4. Define experiment grid
Customize the devices and hidden-layer stacks you want to compare. GPU/TPU runs are skipped automatically if the hardware is unavailable in Colab.

In [ ]:
devices = ['GPU']
architectures = [(64,), (64,32), (64, 32, 16), (64, 32, 16, 8), (64, 32, 16, 4)]
experiment_results = []

## 5. Run training sweeps

In [ ]:
import os
import tensorflow as tf

def get_strategy(device: str):
    device = device.upper()
    if device == "GPU":
        gpus = tf.config.list_physical_devices("GPU")
        if not gpus:
            print("No GPU detected; skipping GPU runs.")
            return None
        return tf.distribute.OneDeviceStrategy(device="/GPU:0")

    if device == "TPU":
        # Detect and initialize TPU
        try:
            tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
            tf.config.experimental_connect_to_cluster(tpu)
            tf.tpu.experimental.initialize_tpu_system(tpu)
            print(f"Running on TPU: {tpu.master()}")
            return tf.distribute.TPUStrategy(tpu)
        except Exception as e:
            print(f"Could not initialize TPU: {e}")
            return None

    # Default CPU
    return tf.distribute.OneDeviceStrategy(device="/CPU:0")




for hidden_units in architectures:
    for device in devices:
        strategy = get_strategy(device)
        if strategy is None:
            continue
        label = format_label(device, hidden_units)
        print(f'\n▶️ Running {label}')
        with strategy.scope():
            model = build_model(input_dim, hidden_units=hidden_units, dropout=0.1, lr=1e-3)
        start = time.time()
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=100,
            verbose=0,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                keras.callbacks.TensorBoard(log_dir=str(PROJECT_ROOT / 'runs' / label)),
            ],
        )
        duration = time.time() - start
        eval_metrics = model.evaluate(val_ds, return_dict=True, verbose=0)
        experiment_results.append(
            {
                'label': label,
                'device': device,
                'hidden_units': hidden_units,
                'history': history,
                'metrics': eval_metrics,
                'duration': duration,
                'avg_epoch_time': duration / len(history.history['loss']),
            }
        )
        plot_history(history, f'{label} (input: {input_dim} features)')
        print(f"Val MAE: {eval_metrics['mae']:.2f} USD — Runtime: {duration:.1f}s")


## 6. Compare devices and architectures

In [ ]:
if not experiment_results:
    raise RuntimeError('No successful runs recorded.')

mae_by_label = {res['label']: res['metrics']['mae'] for res in experiment_results}
time_by_label = {res['label']: res['duration'] for res in experiment_results}
tradeoff_points = {
    res['label']: (res['avg_epoch_time'], res['metrics']['mae']) for res in experiment_results
}

plot_bar(
    mae_by_label,
    'Mean Absolute Error (USD)',
    'Validation Mean Absolute Error (USD) by Device/Architecture',
)
plot_bar(time_by_label, 'Training Time (seconds)', 'Total Training Time per Run (seconds)')
plot_tradeoff(tradeoff_points)

## 7. Record input/output dimensions
Capture the TensorFlow model summary so the report can cite per-layer dimensions.

In [ ]:
reference_model = build_model(input_dim, hidden_units=(128, 64, 32))
reference_model.summary()

With the metrics, plots, and summaries above you can: (1) document input/output dimensions, (2) compare CPU vs GPU execution time, (3) analyze at least three neural-network sizes, and (4) report the best validation MAE/RMSE for the relative accuracy score.

In [ ]:
import json
from pathlib import Path

REPORT_DIR = Path("/kaggle/working/reports/architecture")
HIST_DIR = REPORT_DIR / "histories"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
HIST_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save raw metrics from experiment_results
metrics_path = REPORT_DIR / "metrics.json"
with metrics_path.open("w", encoding="utf-8") as fp:
    json.dump(
        [
            {
                "label": res["label"],
                "device": res["device"],
                "hidden_units": res["hidden_units"],
                "duration_seconds": res["duration"],
                "avg_epoch_time": res["avg_epoch_time"],
                "metrics": res["metrics"],
            }
            for res in experiment_results
        ],
        fp,
        indent=2,
    )
print(f"Metrics saved to {metrics_path}")

# 2. Save history plots per run
for res in experiment_results:
    label = res["label"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(res["history"].history["loss"], label="train")
    axes[0].plot(res["history"].history["val_loss"], label="val")
    axes[0].set_title("Mean Squared Error (USD²) over Epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Mean Squared Error (USD²)")
    axes[0].legend()
    axes[1].plot(res["history"].history["mae"], label="train")
    axes[1].plot(res["history"].history["val_mae"], label="val")
    axes[1].set_title("Mean Absolute Error (USD) over Epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Mean Absolute Error (USD)")
    axes[1].legend()
    fig.suptitle(label)
    fig.tight_layout()
    fig.savefig(HIST_DIR / f"history_{label}.png", dpi=200)
    plt.close(fig)

# 3. Save aggregate comparison plots
plot_bar(
    {res["label"]: res["metrics"]["mae"] for res in experiment_results},
    "Mean Absolute Error (USD)",
    "Validation Mean Absolute Error (USD) by Device/Architecture",
)
plt.savefig(REPORT_DIR / "device_mae.png", dpi=200)
plt.close()

plot_bar(
    {res["label"]: res["duration"] for res in experiment_results},
    "Training Time (seconds)",
    "Total Training Time per Run (seconds)",
)
plt.savefig(REPORT_DIR / "device_runtime.png", dpi=200)
plt.close()

plot_tradeoff(
    {res["label"]: (res["avg_epoch_time"], res["metrics"]["mae"]) for res in experiment_results}
)
plt.savefig(REPORT_DIR / "architecture_tradeoff.png", dpi=200)
plt.close()

# 4. Save model summary text
summary_path = REPORT_DIR / "model_summary.txt"
with summary_path.open("w", encoding="utf-8") as fp:
    reference_model = build_model(input_dim, hidden_units=(128, 64, 32))
    reference_model.summary(print_fn=lambda line: fp.write(line + "\n"))
print(f"Artifacts written to {REPORT_DIR}")


In [ ]:
!cd /kaggle/working && zip -r reports_gpu.zip reports